# 5.1 The Hodgkin–Huxley Model of the Action Potential

*Interactive explorer — Chapter 5, Excitability.*

In the lecture you met the **Hodgkin–Huxley (HH) model**: four equations that describe how the squid giant
axon fires an action potential. This notebook is a virtual lab bench for that model. It doesn't replace the
lecture. It lets you inject current, block channels with drugs and warm the axon up, then check what you
learned against what the model does.

**How to use this notebook**

1. Run everything: **Run ▸ Run All Cells**. The first run in the browser can take up to a minute while
   Python loads.
2. Read the short refresher below, then scroll down to the **explorer**. It has three tabs:
   - **Explore**: sliders and presets that drive the simulation.
   - **Challenges**: questions to work through with the explorer.
   - **Key Ideas**: a summary to check your understanding against.
3. You don't need to write any code. Two optional coding challenges are at the end for anyone who wants to go
   further.
4. Write your answers in the **My notes** cell at the bottom. In JupyterLite your work is kept in this
   browser only, so use **File ▸ Download** to keep a copy.

Model equations and parameter values are adapted from the official code release that accompanies Keener &
Sneyd, *Mathematical Physiology*, 3rd ed. (Springer, 2025): `chap05_excitability/hh.py` (their Fig. 5.6),
released by the authors for reuse at https://github.com/jsneyd/Math-Physiol-3rd-Edition-code.

## A quick refresher

**The membrane is a circuit.** The lipid bilayer stores charge like a capacitor. Ion channels are conducting
pathways through it. The voltage changes only when net current flows onto or off the membrane capacitor:

$$
C_m \frac{dV}{dt} \;=\; -\underbrace{\bar g_{\mathrm{Na}}\, m^3 h\,(V - E_{\mathrm{Na}})}_{I_{\mathrm{Na}}}
\;-\; \underbrace{\bar g_{\mathrm{K}}\, n^4\,(V - E_{\mathrm{K}})}_{I_{\mathrm{K}}}
\;-\; \underbrace{\bar g_{L}\,(V - E_{L})}_{I_{L}} \;+\; I_{\mathrm{app}}
$$

Each ionic current is **conductance × driving force**. The conductance is the maximum ($\bar g$) multiplied
by the fraction of channels that are open. The driving force is how far $V$ is from that ion's reversal
(Nernst) potential $E$.

**Gates.** HH described the fraction of open channels with three "gating variables", each between 0 and 1:

| Gate | Channel | When the membrane depolarises… | Speed | Job in the action potential |
|---|---|---|---|---|
| $m$ | Na⁺ | opens (activation) | **fast** (< 0.5 ms) | drives the upstroke |
| $h$ | Na⁺ | closes (inactivation) | slow (ms) | shuts off Na⁺ entry and causes refractoriness |
| $n$ | K⁺ | opens (activation) | slow (ms) | repolarisation and the after-hyperpolarisation |

Each gate relaxes towards a voltage-dependent target value, $x_\infty(V)$, with a voltage-dependent time
constant, $\tau_x(V)$. In the model this is written with an opening rate $\alpha_x(V)$ and a closing rate
$\beta_x(V)$:
$\;dx/dt = \alpha_x(V)(1-x) - \beta_x(V)\,x$.

**A note on voltages.** Hodgkin and Huxley measured voltage *relative to rest*, and the code keeps that
convention internally (rest = 0 mV). All plots show the absolute membrane potential, taking rest as
−65 mV. In absolute terms $E_{\mathrm{Na}} = +50$ mV, $E_{\mathrm{K}} = -77$ mV and $E_L \approx -54$ mV.
Currents are in µA/cm², conductances in mS/cm² and time in ms. Inward currents are negative.

In [ ]:
# Setup: in JupyterLite (in-browser Python) the widget library has to be installed first.
import sys
if "pyodide" in sys.modules:
    %pip install -q ipywidgets

import math
import warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# The in-browser matplotlib prints harmless deprecation warnings when drawing; hide them.
warnings.filterwarnings('ignore', category=matplotlib.MatplotlibDeprecationWarning)

In [ ]:
# ---- Parameters: Hodgkin & Huxley (1952), squid giant axon at 6.3 °C ----
# Voltages are relative to rest (rest = 0 mV), as in the original paper.
gna, gk, gl = 120.0, 36.0, 0.3       # maximal conductances (mS/cm^2)
vna, vk, vl = 115.0, -12.0, 10.6     # reversal potentials relative to rest (mV)
cm = 1.0                             # membrane capacitance (uF/cm^2)
V_REST = -65.0                       # absolute resting potential, used only for plotting (mV)

# ---- Opening (alpha) and closing (beta) rates of the gates (1/ms) ----
def _ratio(x, y):
    # x / (exp(x/y) - 1), with the 0/0 case at x = 0 handled safely
    return y * (1.0 - x / (2.0 * y)) if abs(x / y) < 1e-6 else x / (math.exp(x / y) - 1.0)

def alpham(v): return 0.1 * _ratio(25.0 - v, 10.0)
def betam(v):  return 4.0 * math.exp(-v / 18.0)
def alphah(v): return 0.07 * math.exp(-v / 20.0)
def betah(v):  return 1.0 / (math.exp((30.0 - v) / 10.0) + 1.0)
def alphan(v): return 0.01 * _ratio(10.0 - v, 10.0)
def betan(v):  return 0.125 * math.exp(-v / 80.0)

# Steady-state gate values at rest: the starting point of every simulation
def x_inf(alpha, beta, v): return alpha(v) / (alpha(v) + beta(v))
m0, h0, n0 = x_inf(alpham, betam, 0.0), x_inf(alphah, betah, 0.0), x_inf(alphan, betan, 0.0)

### The gates at a glance

The left panel shows where each gate settles if the voltage is held fixed ($x_\infty$). The right panel
shows how fast it gets there ($\tau_x$). Compare $\tau_m$ with $\tau_h$ and $\tau_n$: that difference in
timing is what makes an action potential possible.

In [ ]:
V_rel = np.linspace(-40, 120, 400)          # relative to rest
V_abs = V_rel + V_REST
gates = {'m (Na⁺ activation)':   (alpham, betam, 'tab:red'),
         'h (Na⁺ inactivation)': (alphah, betah, 'tab:orange'),
         'n (K⁺ activation)':    (alphan, betan, 'tab:blue')}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.6))
for label, (a, b, c) in gates.items():
    al = np.array([a(v) for v in V_rel]); be = np.array([b(v) for v in V_rel])
    ax1.plot(V_abs, al / (al + be), color=c, lw=2, label=label)
    ax2.plot(V_abs, 1 / (al + be), color=c, lw=2, label=label)
for ax in (ax1, ax2):
    ax.axvline(V_REST, color='grey', ls=':', lw=1)
    ax.set_xlabel('Membrane potential (mV)')
ax1.set_ylabel('Steady-state value $x_\\infty$'); ax1.set_title('Where the gate settles')
ax2.set_ylabel('Time constant $\\tau_x$ (ms)');   ax2.set_title('How fast it gets there')
ax1.legend(fontsize=9); ax1.text(V_REST + 2, 0.92, 'rest', color='grey')
plt.tight_layout(); plt.show()

### The original simulation (Keener & Sneyd, Fig. 5.6)

This is the book's simulation. No current is injected. Instead the membrane starts **7 mV depolarised** from
rest, with all the gates at their resting values. Watch what happens to that small kick.

In [ ]:
def deriv(t, y):
    v, n, m, h = y
    vd = (1.0 / cm) * (-gk * n**4 * (v - vk) - gna * m**3 * h * (v - vna) - gl * (v - vl))
    nd = alphan(v) * (1.0 - n) - betan(v) * n
    md = alpham(v) * (1.0 - m) - betam(v) * m
    hd = alphah(v) * (1.0 - h) - betah(v) * h
    return [vd, nd, md, hd]

y0 = [7.0, n0, m0, h0]                       # start 7 mV above rest
tout = np.linspace(0, 50, 500)
soln = solve_ivp(deriv, [0, 50], y0, method='Radau', t_eval=tout, rtol=1e-6, atol=1e-9)

plt.figure(figsize=(7, 3.2))
plt.plot(soln.t, soln.y[0] + V_REST, 'k', lw=2)
plt.axhline(V_REST, color='grey', ls=':', lw=1)
plt.xlabel('Time (ms)'); plt.ylabel('Membrane potential (mV)')
plt.title('A 7 mV kick from rest triggers a full action potential')
plt.tight_layout(); plt.show()

## The explorer

Run the next cell to build the explorer. The **Explore** tab has two kinds of control:

- **Presets** load a ready-made experiment. Many of the challenges start from one.
- **Sliders** change the experiment. The simulation reruns when you release a slider. You can also click a
  slider's number and type an exact value.

The plots show, from top to bottom: the membrane potential; the Na⁺ and K⁺ conductances (or currents);
the three gates; and the current you inject. Shaded bars mark the stimulus pulses. The dashed lines in the
top panel mark $E_{\mathrm{Na}}$ and $E_{\mathrm{K}}$.

The simulation steps forward 0.01 ms at a time (a fixed-step Runge–Kutta method). That is fast enough to feel
interactive in the browser.

In [ ]:
def simulate(I1=10.0, dur1=1.0, pulse2=False, delay2=15.0, I2=10.0, Ihold=0.0,
             gna_pct=100.0, gk_pct=100.0, temp=6.3, t_end=50.0, t_on=5.0):
    # Simulate the HH model. Stimulus pulse 1 starts at t_on. Pulse 2 starts delay2 ms after pulse 1.
    # The sustained step (Ihold) runs from t_on to the end. Returns a dict of time courses.
    phi = 3.0 ** ((temp - 6.3) / 10.0)           # temperature speeds up all gates (Q10 = 3)
    gNa, gK = gna * gna_pct / 100.0, gk * gk_pct / 100.0
    dt = 0.01 / max(1.0, phi / 4.0)              # smaller steps when the gates get very fast
    N = int(round(t_end / dt))

    def stim(t):
        I = Ihold if t >= t_on else 0.0
        if t_on <= t < t_on + dur1:
            I += I1
        if pulse2 and t_on + delay2 <= t < t_on + delay2 + dur1:
            I += I2
        return I

    def f(t, v, n, m, h):
        dv = (stim(t) - gK * n**4 * (v - vk) - gNa * m**3 * h * (v - vna) - gl * (v - vl)) / cm
        dn = phi * (alphan(v) * (1.0 - n) - betan(v) * n)
        dm = phi * (alpham(v) * (1.0 - m) - betam(v) * m)
        dh = phi * (alphah(v) * (1.0 - h) - betah(v) * h)
        return dv, dn, dm, dh

    out = np.empty((N + 1, 4)); out[0] = y = (0.0, n0, m0, h0)
    for i in range(N):                           # classic 4th-order Runge-Kutta
        t = i * dt
        k1 = f(t, *y)
        k2 = f(t + dt/2, *[a + dt/2 * b for a, b in zip(y, k1)])
        k3 = f(t + dt/2, *[a + dt/2 * b for a, b in zip(y, k2)])
        k4 = f(t + dt,   *[a + dt * b for a, b in zip(y, k3)])
        y = tuple(a + dt/6 * (b1 + 2*b2 + 2*b3 + b4) for a, b1, b2, b3, b4 in zip(y, k1, k2, k3, k4))
        out[i + 1] = y

    t = np.arange(N + 1) * dt
    v, n, m, h = out.T
    g_na, g_k = gNa * m**3 * h, gK * n**4
    return dict(t=t, V=v + V_REST, n=n, m=m, h=h, gNa=g_na, gK=g_k,
                INa=g_na * (v - vna), IK=g_k * (v - vk), Istim=np.array([stim(s) for s in t]))

def spike_times(r, threshold=-20.0):
    # Times (ms) at which V crosses the threshold (mV) going upwards
    V = r['V']
    up = np.where((V[:-1] < threshold) & (V[1:] >= threshold))[0]
    return r['t'][up]

In [ ]:
import ipywidgets as W

STYLE = {'description_width': '200px'}
LAYOUT = W.Layout(width='440px')

def slider(desc, value, lo, hi, step, fmt='.1f'):
    return W.FloatSlider(value=value, min=lo, max=hi, step=step, description=desc, readout_format=fmt,
                         continuous_update=False, style=STYLE, layout=LAYOUT)

ctrl = dict(
    I1=slider('Pulse 1 amplitude (µA/cm²)', 10, -50, 100, 0.5),
    dur1=slider('Pulse duration (ms)', 1.0, 0.1, 10, 0.1),
    pulse2=W.Checkbox(value=False, description='Add a second pulse', indent=False),
    delay2=slider('Pulse 2 starts after (ms)', 15, 1, 40, 0.5),
    I2=slider('Pulse 2 amplitude (µA/cm²)', 10, -50, 100, 0.5),
    Ihold=slider('Sustained current (µA/cm²)', 0, -10, 200, 0.5),
    gna_pct=slider('Na⁺ channels available (%)', 100, 0, 100, 1, '.0f'),
    gk_pct=slider('K⁺ channels available (%)', 100, 0, 100, 1, '.0f'),
    temp=slider('Temperature (°C)', 6.3, 0, 40, 0.1),
    t_end=slider('Simulation length (ms)', 50, 20, 200, 5, '.0f'),
)
DEFAULTS = {k: c.value for k, c in ctrl.items()}

PRESETS = {
    'Single spike (default)':              {},
    'Below threshold':                     {'I1': 2},
    'Two pulses: refractory period':       {'pulse2': True, 'delay2': 20},
    'TTX: block Na⁺ channels':             {'gna_pct': 0},
    'TEA: block K⁺ channels':              {'gk_pct': 0, 't_end': 100},
    'Repetitive firing':                   {'I1': 0, 'Ihold': 15, 't_end': 100},
    'Anode-break excitation':              {'I1': -15, 'dur1': 5},
    'Depolarisation block':                {'I1': 0, 'Ihold': 180, 't_end': 100},
    'Warm axon (20 °C)':                   {'temp': 20},
}
preset = W.Dropdown(options=list(PRESETS), description='Preset experiment', style=STYLE, layout=LAYOUT)
middle = W.ToggleButtons(options=['Conductances', 'Currents'], description='Second panel shows',
                         style={'description_width': '130px', 'button_width': '110px'})
readout = W.HTML()
plot_out = W.Output()
_loading_preset = False

def draw(r, show):
    fig, ax = plt.subplots(4, 1, figsize=(9, 8.5), sharex=True,
                           gridspec_kw={'height_ratios': [3, 2, 2, 1]})
    on = r['Istim'] != 0
    for a in ax[:3]:                                  # shade the stimulus
        a.fill_between(r['t'], 0, 1, where=on, color='gold', alpha=0.25, lw=0,
                       transform=a.get_xaxis_transform())
    ax[0].plot(r['t'], r['V'], 'k', lw=2)
    for E, name, c in ((vna + V_REST, '$E_{Na}$', 'tab:red'), (vk + V_REST, '$E_K$', 'tab:blue')):
        ax[0].axhline(E, color=c, ls='--', lw=1)
        ax[0].text(r['t'][-1], E, ' ' + name, color=c, va='center')
    ax[0].set_ylim(-100, 65); ax[0].set_ylabel('V (mV)')
    if show == 'Conductances':
        ax[1].plot(r['t'], r['gNa'], color='tab:red', lw=2, label='$g_{Na}$')
        ax[1].plot(r['t'], r['gK'], color='tab:blue', lw=2, label='$g_K$')
        ax[1].set_ylabel('g (mS/cm²)')
    else:
        ax[1].plot(r['t'], r['INa'], color='tab:red', lw=2, label='$I_{Na}$')
        ax[1].plot(r['t'], r['IK'], color='tab:blue', lw=2, label='$I_K$')
        ax[1].axhline(0, color='grey', lw=0.8)
        ax[1].set_ylabel('I (µA/cm²)\n← inward | outward →')
    ax[1].legend(loc='upper right')
    for x, c in (('m', 'tab:red'), ('h', 'tab:orange'), ('n', 'tab:blue')):
        ax[2].plot(r['t'], r[x], color=c, lw=2, label=x)
    ax[2].set_ylim(-0.03, 1.03); ax[2].set_ylabel('gate'); ax[2].legend(loc='upper right', ncol=3)
    ax[3].plot(r['t'], r['Istim'], color='goldenrod', lw=2)
    ax[3].set_ylabel('$I_{app}$\n(µA/cm²)'); ax[3].set_xlabel('Time (ms)')
    ax[3].set_xlim(0, r['t'][-1])
    fig.align_ylabels(); plt.tight_layout(); plt.show()

def update(*_):
    if _loading_preset:
        return
    vals = {k: c.value for k, c in ctrl.items()}
    for k in ('delay2', 'I2'):
        ctrl[k].disabled = not vals['pulse2']
    readout.value = '<i>Simulating…</i>'
    r = simulate(**vals)
    st = spike_times(r)
    txt = f'<b>Spikes:</b> {len(st)} &nbsp;|&nbsp; <b>Peak V:</b> {r["V"].max():.1f} mV'
    if len(st) >= 3:
        txt += f' &nbsp;|&nbsp; <b>Firing rate:</b> {1000 / np.diff(st[1:]).mean():.0f} Hz'
    readout.value = txt
    with plot_out:
        plot_out.clear_output(wait=True)
        draw(r, middle.value)

def load_preset(change):
    global _loading_preset
    _loading_preset = True
    for k, v in {**DEFAULTS, **PRESETS[change['new']]}.items():
        ctrl[k].value = v
    _loading_preset = False
    update()

for c in ctrl.values():
    c.observe(update, 'value')
middle.observe(update, 'value')
preset.observe(load_preset, 'value')

def box(title, keys):
    return W.VBox([W.HTML(f'<b>{title}</b>')] + [ctrl[k] for k in keys],
                  layout=W.Layout(padding='4px 12px 4px 0'))

explore = W.VBox([
    W.HTML('<p style="max-width:860px">Start with a preset, then change one slider at a time. For every '
           'change, ask: <i>which gate is responsible?</i> The gates panel (m, h, n) usually tells you.</p>'),
    W.HBox([preset, middle]),
    W.HBox([box('Stimulus', ['I1', 'dur1', 'pulse2', 'delay2', 'I2', 'Ihold']),
            box('Axon', ['gna_pct', 'gk_pct', 'temp', 't_end'])], layout=W.Layout(flex_flow='row wrap')),
    readout,
    plot_out,
])

In [ ]:
CHALLENGES_HTML = r'''<div style="max-width:860px; line-height:1.5">
<p>Work through these with the <b>Explore</b> tab. Write your answers in the <b>My notes</b> cell at the
bottom of the notebook. Each challenge says which preset to start from. Hints are hidden; try first.</p>

<h3>1. Is there a threshold?</h3>
<p><i>Start:</i> <b>Below threshold</b>. Slowly increase <b>Pulse 1 amplitude</b>.</p>
<ol type="a">
<li>Find the smallest amplitude that triggers an action potential with a 1 ms pulse.</li>
<li>Above that amplitude, double and then triple the stimulus. How much do the height and shape of the
spike change? What does "all-or-none" mean here?</li>
<li>Look at the m and h gates just below and just above threshold. What is different?</li>
</ol>
<details><summary>Hint</summary>The threshold for a 1 ms pulse is between 2 and 10 µA/cm². Near threshold,
watch how long the voltage hesitates before it either takes off or falls back. Above threshold the upstroke is
driven by the Na⁺ channels themselves (positive feedback), not by your stimulus.</details>

<h3>2. Strength and duration</h3>
<p><i>Start:</i> <b>Single spike</b>. Repeat Challenge 1a for pulse durations of 0.2, 0.5, 1, 2 and 5 ms.</p>
<ol type="a">
<li>Make a table of duration and threshold amplitude, then sketch the curve.</li>
<li>Why does a short pulse need a bigger current? Think about the membrane capacitor and how much charge you
deliver (current × time).</li>
</ol>
<details><summary>Hint</summary>To reach threshold you have to put a certain amount of charge on the membrane
before it leaks away. The optional coding challenge A at the end plots this curve automatically, so you can
check your table.</details>

<h3>3. Refractory periods</h3>
<p><i>Start:</i> <b>Two pulses: refractory period</b>. Shorten <b>Pulse 2 starts after</b> step by step.</p>
<ol type="a">
<li>Below what interval does the second pulse fail to produce a spike? Now raise <b>Pulse 2 amplitude</b>.
Can you rescue the spike?</li>
<li>Find the interval below which <i>no</i> amplitude (up to 100 µA/cm²) works. That is the absolute
refractory period. The range in which a bigger stimulus still works is the relative refractory period.</li>
<li>Use the gates panel to explain both. What are h and n doing at the moment of the second pulse?</li>
<li>Compare the size of second spikes you can rescue with the first spike. Why are they smaller?</li>
</ol>
<details><summary>Hint</summary>After a spike, h is low (Na⁺ channels inactivated) and n is high
(K⁺ channels still open). Both make a second spike harder. They recover at different rates.</details>

<h3>4. Pharmacology: TTX and TEA</h3>
<p><i>Start:</i> <b>TTX: block Na⁺ channels</b>, then <b>TEA: block K⁺ channels</b>.</p>
<ol type="a">
<li>With 0 % Na⁺ channels, can <i>any</i> stimulus produce an action potential? Now try 50 % and 25 %.
Does the spike shrink gradually or fail all at once? What happens to the threshold?
<br><i>Careful:</i> the spike counter only checks whether V crosses −20 mV, and a very large stimulus can
push V that high on its own. Use the conductance panel to decide whether a response is a real action
potential.</li>
<li>With K⁺ channels blocked, what happens after the upstroke? Which gate is now the only thing ending
the spike, and why isn't it enough to bring V back to rest?</li>
<li>Local anaesthetics such as lidocaine block Na⁺ channels. Use your answer to (a) to explain why they
stop pain signals.</li>
</ol>
<details><summary>Hint</summary>In (b), look at the h gate and at the leak. With no K⁺ current, only Na⁺
inactivation and the small leak current oppose the depolarisation.</details>

<h3>5. Coding stimulus strength as firing rate</h3>
<p><i>Start:</i> <b>Repetitive firing</b>.</p>
<ol type="a">
<li>Vary the <b>Sustained current</b> between 0 and 100 µA/cm² and note the firing rate (shown above the
plots). Does the rate increase smoothly from 0 Hz, or does firing start suddenly at some minimum rate?</li>
<li>Load <b>Depolarisation block</b>. What happens to firing when the current is very large? Which gate
explains it?</li>
<li>Real sensory neurons encode stimulus intensity in their firing rate. What range of intensities could
this axon encode?</li>
</ol>
<details><summary>Hint</summary>HH-type neurons have a minimum firing rate: they jump from silence straight to
tens of Hz. At very high currents the membrane stays depolarised, so h never recovers.</details>

<h3>6. Anode-break excitation</h3>
<p><i>Start:</i> <b>Anode-break excitation</b>. The stimulus here is <i>hyperpolarising</i> (negative).</p>
<ol type="a">
<li>When does the spike occur: during the pulse or after it ends?</li>
<li>Using the gates panel, explain how <i>hyperpolarising</i> a neuron can make it fire. What happens to h and
n during the pulse?</li>
<li>Find the weakest hyperpolarising pulse (5 ms) that still causes a spike.</li>
</ol>
<details><summary>Hint</summary>During hyperpolarisation, h rises above its resting value (more Na⁺ channels
become available) and n falls (fewer K⁺ channels open). These gates are slow, so they are still in that state
when V jumps back up at the end of the pulse.</details>

<h3>7. Temperature</h3>
<p><i>Start:</i> <b>Single spike</b>. Change <b>Temperature</b> from 6.3 °C (HH's experiments) to 15, 25 and
35 °C. Raise the stimulus if needed.</p>
<ol type="a">
<li>How do the width and height of the spike change? Why?</li>
<li>Above about 35 °C the response shrinks to a small bump, however hard you stimulate. This is called
heat block. Use the timing of m compared with h and n to suggest why it happens.</li>
</ol>
<details><summary>Hint</summary>In this model warming speeds up <i>all</i> gates by the same factor
(Q10 = 3). The leak and the membrane capacitor do not speed up. Ask whether the upstroke can still get going
before h and n shut it down.</details>
</div>'''

KEY_IDEAS_HTML = r'''<div style="max-width:860px; line-height:1.5">
<h3>Key ideas</h3>
<ul>
<li><b>Positive feedback makes the upstroke.</b> Depolarisation opens Na⁺ channels (m↑). Na⁺ flows in, which
depolarises the membrane further. Once this loop takes off, the spike no longer depends on the stimulus.
That is why the action potential is all-or-none.</li>
<li><b>Two slower negative feedbacks end it.</b> Na⁺ inactivation (h↓) and K⁺ activation (n↑) bring V back
down. The key is the <i>timing</i>: m is fast, while h and n are slow.</li>
<li><b>The peak approaches E<sub>Na</sub> and the undershoot approaches E<sub>K</sub>.</b> The membrane
potential moves towards the reversal potential of whichever conductance dominates at that moment.</li>
<li><b>Threshold is not a fixed voltage.</b> It depends on how fast you depolarise (strength–duration) and on
recent history (refractoriness, anode break). What matters is whether Na⁺ activation can outrun the restoring
processes.</li>
<li><b>Refractoriness comes from slow recovery.</b> h needs time to recover and n needs time to fall. This
limits the maximum firing rate, and in an axon it stops an action potential from travelling backwards.</li>
<li><b>Stimulus strength is encoded as firing rate</b>, within limits. The HH axon has a minimum firing rate,
and it falls silent again (depolarisation block) if the current is too large.</li>
<li><b>Drugs act on specific gates.</b> TTX and local anaesthetics remove the Na⁺ positive feedback. TEA removes
the K⁺ repolarisation. Each produces a distinct, predictable change in the action potential.</li>
<li><b>Temperature matters.</b> Gating speeds up about 3-fold per 10 °C, which gives briefer action
potentials. Mammalian neurons work at 37 °C with different channels.</li>
</ul>
<h3>What this model leaves out</h3>
<ul>
<li>Space: this is a single patch of membrane. How the action potential <i>travels</i> along the axon is the
topic of Chapter 6 (Waves).</li>
<li>Many channel types found in real neurons, such as Ca²⁺ channels, Ca²⁺-activated K⁺ channels and
A-type K⁺ channels.</li>
<li>Ion concentrations stay fixed. In reality, prolonged firing slowly changes them.</li>
</ul>
<p>Despite this, HH worked out these gating rules from voltage-clamp data in 1952, decades before anyone could
see an ion channel. The model is a successful <i>hypothesis</i> about molecular machinery.</p>
</div>'''

In [ ]:
tabs = W.Tab(children=[explore, W.HTML(CHALLENGES_HTML), W.HTML(KEY_IDEAS_HTML)])
for i, title in enumerate(['Explore', 'Challenges', 'Key Ideas']):
    tabs.set_title(i, title)
display(tabs)
update()

## Optional coding challenges

These cells reuse the `simulate()` function from above. You can run them as they are, then change the
numbers marked `# <- try changing`.

### A. Strength–duration curve

For each pulse duration, the code searches for the threshold amplitude by repeated halving (bisection).
This takes about 10–30 s in the browser.

In [ ]:
durations = [0.2, 0.5, 1, 2, 5, 10]          # <- try changing (ms)

def fires(I, dur):
    return len(spike_times(simulate(I1=I, dur1=dur, t_end=dur + 20))) > 0

thresholds = []
for dur in durations:
    lo, hi = 0.0, 200.0                       # threshold lies somewhere in between
    for _ in range(10):
        mid = (lo + hi) / 2
        lo, hi = (lo, mid) if fires(mid, dur) else (mid, hi)
    thresholds.append(hi)
    print(f'{dur:5.1f} ms pulse -> threshold ≈ {hi:6.2f} µA/cm²')

plt.figure(figsize=(6, 3.5))
plt.plot(durations, thresholds, 'o-', lw=2)
plt.xlabel('Pulse duration (ms)'); plt.ylabel('Threshold amplitude (µA/cm²)')
plt.title('Strength–duration curve'); plt.tight_layout(); plt.show()

**Questions.** Does the curve level off for long pulses? That minimum current is called the *rheobase*.
How does it compare with the sustained current needed for repetitive firing in Challenge 5?

### B. Firing rate vs. current (f–I curve)

This code runs a 200 ms sustained current step at several amplitudes and measures the firing rate. It takes
about 30–60 s in the browser.

In [ ]:
currents = [0, 2, 4, 6, 8, 10, 15, 20, 40, 60, 80, 100, 150, 200]   # <- try changing (µA/cm²)

rates = []
for I in currents:
    st = spike_times(simulate(I1=0, Ihold=I, t_end=200))
    st = st[st > 50]                          # ignore the first 50 ms (onset transient)
    rates.append(1000 / np.diff(st).mean() if len(st) >= 2 else 0.0)

plt.figure(figsize=(6, 3.5))
plt.plot(currents, rates, 'o-', lw=2)
plt.xlabel('Sustained current (µA/cm²)'); plt.ylabel('Firing rate (Hz)')
plt.title('f–I curve of the HH model'); plt.tight_layout(); plt.show()

**Questions.** Where does sustained firing start? Is the jump from 0 Hz sudden or gradual? Where does firing
stop, and why? Try adding currents between those you already have to find these points more precisely.

## My notes

*Double-click this cell to edit it. Write your answers to the challenges here.*

**1. Threshold:**

**2. Strength–duration:**

**3. Refractory periods:**

**4. TTX / TEA:**

**5. Firing rate:**

**6. Anode break:**

**7. Temperature:**